<div style="display: flex; justify-content: space-between; align-items: flex-start; border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: -8px;">
    <div style="width: 50%;">
        <h2>
            <span style="color: #B30033;">▍</span>TRABAJO DE FIN DE MÁSTER
        </h2>
        <h1 style="margin-top: -10px;">MARCO DE TRABAJO DE RECOMENDACIÓN DE PERSONAL PARA LA FORMACIÓN DE EQUIPOS EN PROYECTOS DE INGENIERÍA DEL SOFTWARE</h1>
    </div>
    <div style="width: 50%; text-align: right;">
        <div style="display: flex; justify-content: space-between; align-items: flex-start; margin-top: 30px;">
            <div style="width: 20%;"></div>
            <div style="width: 80%; border-left: 2px solid #555555; padding-left: 20px;">
                <div style="margin-bottom: 20px;">
                    <p style="margin: 0; font-size: 1.4em; font-weight: bold;">
                        Tutores
                    </p>
                </div>
                <div style="margin-top: 8px; text-align: right;">
                    <span style="display: block; font-size: 1em; color: #D0D0D0;">
                        Aurora Vizcaíno Barcelo
                    </span>
                    <span style="display: block; font-size: 1em; color: #D0D0D0; margin-top: 5px;">
                        Julio Alberto López Gómez
                    </span>
                </div>
            </div>
        </div>
    </div>
</div>

<div style="margin-top: 30px; padding: 20px; background-color: rgba(179, 0, 51, 0.05); border-left: 4px solid #B30033; border-radius: 4px;">
    <h3 style="margin-top: 0; color: #B30033;">🎯 Objetivo de esta Libreta (Fase 4: Evaluación — contraste con las reglas expertas)</h3>
    <p style="font-size: 1.1em; line-height: 1.6; color: #E0E0E0;">
        El objetivo de esta libreta es contrastar las <strong>reglas de asociación descubiertas</strong> en la Libreta anterior contra las <strong>27 reglas definidas por la experta de dominio</strong> Aurora Vizcaíno Barceló, para validar en qué medida los patrones encontrados en los datos coinciden, contradicen o carecen de evidencia respecto al conocimiento experto.
    </p>
    <ul style="font-size: 1.1em; line-height: 1.6; color: #D0D0D0; margin-bottom: 0;">
        <li><strong>Carga:</strong> Primero cargamos las reglas descubiertas en la libreta 3 (<code>reglas_limpias_soft.csv</code>) y las reglas expertas estructuradas, las cuales he ordenado en un archivo JSON para poder hacer más fácil su tratamiento (<code>reglas_expertas_riesgo.json</code>).</li>
        <li><strong>Clasificación de alcance:</strong> de las 27 reglas expertas, solo una parte es evaluable con los datos de los que disponemos. Esto es porque el resto combina condiciones de nivel equipo/proyecto (o de hard skills), que se evaluarán en fases posteriores del framework.</li>
        <li><strong>Contraste:</strong> para cada regla experta con componente de soft skills evaluable, buscamos la evidencia en las distintas reglas descubiertas y las clasificamos en base a si esta coincide, se contradice o si no tenemos evidencias suficientes.</li>
        <li><strong>Conclusiones:</strong> Estudiaremos la cobertura y los hallazgos relevantes para el objetivo central del proyecto.</li>
    </ul>
</div>
<br/>

<div style="border-bottom: 2px solid #555555; padding-bottom: 25px; margin-bottom: 10px">
    <div style="display: flex; align-items: center; margin-bottom: 10px;">
        <span style="color: #B30033; font-size: 1.5em; margin-right: 10px;">▍</span>
        <h3 style="margin: 0; font-size: 1.4em; font-weight: bold">
            Realizado por Miguel Ángel Sánchez-Medina Camacho <span style="font-size: 0.7em; font-style: italic; font-weight: normal;"></span>
        </h3>
    </div>
</div>

### 4.1. Configuración del entorno y carga de datos

En esta primera celda cargamos las reglas de asociación depuradas de la Libreta 3 (`reglas_limpias_soft.csv`, las 9.575 reglas que ya no tienen nulos ni circularidad de los proxies) y el archivo semiestructurado con las reglas expertas (`reglas_expertas_riesgo.json`), que clasifica cada una de las 27 reglas del documento en `soft_parcial`, `hard_parcial` o `proyecto_json` según su nivel de evaluación.

In [1]:
import os
import sys
import json
import pandas as pd

sys.path.append(os.path.abspath(os.path.join('../..')))
from src.soft_skills import config

reglas_limpias = pd.read_csv(config.REGLAS_LIMPIAS_PATH)
reglas_limpias['items_totales'] = reglas_limpias.apply(lambda row: set(str(row['antecedents']).split(' | ')) | set(str(row['consequents']).split(' | ')),axis=1)

with open(config.REGLAS_EXPERTAS_PATH, encoding='utf-8') as f:
    reglas_experto = json.load(f)['reglas']

reglas_soft = [r for r in reglas_experto if r['tipo'] == 'soft_parcial']

print(f"Reglas descubiertas en libreta 3: {len(reglas_limpias)}")
print(f"Reglas expertas totales: {len(reglas_experto)}")
print(f"Reglas expertas con componente soft evaluable: {len(reglas_soft)}")

Reglas descubiertas en libreta 3: 9575
Reglas expertas totales: 27
Reglas expertas con componente soft evaluable: 9


### 4.2. Función de Búsqueda de evidencia

Para cada una de las reglas expertas con componentes de soft skills evaluables, vamos a comprobar si sus ítems asociados **coexisten** en alguna regla descubierta (`reglas_limpias`). El motivo de esto es saber si el algoritmo encontró esa combinación como suficientemente frecuente y fiable (soporte ≥0.12 y confianza ≥0.6, ya aplicados en la Libreta 3).

In [2]:
def buscar_evidencia(items_esperados, df_reglas=reglas_limpias, top_n=5):
    items_esperados = set(items_esperados)
    masc = df_reglas['items_totales'].apply(lambda s: items_esperados.issubset(s))
    resultado = df_reglas[masc].sort_values('lift', ascending=False)
    return resultado.head(top_n)

### 4.3. Contraste de las diferentes reglas evaluables

Una vez ya tenemos todo lo necesario, para este paso vamos a recorrer las reglas expertas con componentes de soft skills para así buscar evidencias o combinaciones de ítems en las reglas de la libreta anterior que ya tenemos limpias. A partir de ahí, veremos qué obtenemos e interpretaremos los resultados manualmente más adelante.

In [3]:
for r in reglas_soft:
    print(f"\n{'='*70}")
    print(f"Regla experta #{r['id']}: {r['riesgo']}")
    print(f"Ítems localizados: {r['items_soft']}")
    if 'nota' in r:
        print(f"Nota: {r['nota']}")
    evidencia = buscar_evidencia(r['items_soft'])
    if evidencia.empty:
        print("→ SIN EVIDENCIA en reglas_limpias_soft (no coexisten con confianza≥0.6)")
    else:
        display(evidencia[['antecedents', 'consequents', 'support', 'confidence', 'lift']])


Regla experta #8: Sobrecarga de trabajo
Ítems localizados: ['Neuroticismo_Nivel_Alto', 'WorkWeekHrs_Bin_>55h']
Nota: La condición de más de 2 proyectos concurrentes no es evaluable con datos de candidato individual (no existe en la encuesta de 2019).
→ SIN EVIDENCIA en reglas_limpias_soft (no coexisten con confianza≥0.6)

Regla experta #10: Burnout
Ítems localizados: ['Neuroticismo_Nivel_Alto', 'WorkWeekHrs_Bin_45-55h']
Nota: La condición de bajo solapamiento horario con comunicación síncrona requerida no es evaluable con datos de candidato individual.
→ SIN EVIDENCIA en reglas_limpias_soft (no coexisten con confianza≥0.6)

Regla experta #11: Aislamiento social
Ítems localizados: ['WorkRemote_Bucket_Remoto Alto (>70%)']
Nota: Solo la condición de trabajo remoto >70% es evaluable a nivel de candidato; las otras tres condiciones son atributos de proyecto/equipo.
→ SIN EVIDENCIA en reglas_limpias_soft (no coexisten con confianza≥0.6)

Regla experta #12: Riesgo de conflictos
Ítems localiz

,antecedents,consequents,support,confidence,lift
533,Amabilidad_Nivel_Bajo | Employment_Employed fu...,WorkRemote_Bucket_Presencial / Bajo remoto (<30%),0.131880,0.897715,1.481859
745,Amabilidad_Nivel_Bajo | WorkLoc_Office,Employment_Employed full-time | WorkRemote_Buc...,0.131880,0.810242,1.449885
804,Amabilidad_Nivel_Bajo | WorkLoc_Office,WorkRemote_Bucket_Presencial / Bajo remoto (<30%),0.142378,0.874740,1.443935
923,Amabilidad_Nivel_Bajo | WorkWeekHrs_Bin_35-45h,Employment_Employed full-time | WorkRemote_Buc...,0.162844,0.799303,1.430310
1168,Amabilidad_Nivel_Bajo | Employment_Employed fu...,WorkWeekHrs_Bin_35-45h,0.162844,0.767397,1.407807



Regla experta #22: Falta de soporte para el trabajo remoto
Ítems localizados: ['WorkRemote_Bucket_Remoto Alto (>70%)']
Nota: La condición de modo no presencial es evaluable (WorkRemote_Bucket distinto de 'Presencial / Bajo remoto'); la existencia de políticas de empresa no lo es a nivel de candidato.
→ SIN EVIDENCIA en reglas_limpias_soft (no coexisten con confianza≥0.6)

Regla experta #23: Resistencia al cambio
Ítems localizados: ['Apertura_Nivel_Bajo']
Nota: Solo la condición de baja apertura a la experiencia es evaluable a nivel de candidato individual.


,antecedents,consequents,support,confidence,lift
2753,Apertura_Nivel_Bajo | Employment_Employed full...,WorkWeekHrs_Bin_35-45h,0.127847,0.727751,1.335074
3010,Apertura_Nivel_Bajo | Employment_Employed full...,WorkRemote_Bucket_Presencial / Bajo remoto (<30%),0.140846,0.801744,1.323440
3736,Apertura_Nivel_Bajo | WorkWeekHrs_Bin_35-45h,Employment_Employed full-time,0.127847,0.939492,1.295027
3987,Apertura_Nivel_Bajo | WorkRemote_Bucket_Presen...,Employment_Employed full-time,0.140846,0.932925,1.285976
6769,Apertura_Nivel_Bajo,Employment_Employed full-time | WorkRemote_Buc...,0.140846,0.607698,1.087443



Regla experta #24: Fatiga digital
Ítems localizados: ['WorkRemote_Bucket_Remoto Alto (>70%)', 'WorkChallenge_Meetings']
Nota: Trabajo remoto alto y percepción de exceso de reuniones son evaluables a nivel de candidato; la existencia de política de desconexión es un atributo de proyecto/empresa.
→ SIN EVIDENCIA en reglas_limpias_soft (no coexisten con confianza≥0.6)

Regla experta #25: Burnout
Ítems localizados: ['WorkRemote_Bucket_Remoto Alto (>70%)', "WorkPlan_There's no schedule or spec; I work on what seems most important or urgent"]
Nota: Trabajo remoto alto y ausencia de horario/plan definido (WorkPlan) son evaluables a nivel de candidato; política de desconexión y ajuste de plazos son atributos de proyecto.
→ SIN EVIDENCIA en reglas_limpias_soft (no coexisten con confianza≥0.6)

Regla experta #26: Burnout
Ítems localizados: ['WorkChallenge_Meetings']
Nota: Componente débil: solo la percepción individual de exceso de reuniones es evaluable; 'múltiples equipos distribuidos' es un 

,antecedents,consequents,support,confidence,lift
262,WorkChallenge_Meetings | WorkRemote_Bucket_Pre...,Employment_Employed full-time | WorkLoc_Office,0.137455,0.632804,1.527395
286,WorkChallenge_Meetings | WorkLoc_Office,Employment_Employed full-time | WorkRemote_Buc...,0.137455,0.849850,1.520761
332,Employment_Employed full-time | WorkChallenge_...,WorkRemote_Bucket_Presencial / Bajo remoto (<30%),0.137455,0.916692,1.513184
478,WorkChallenge_Meetings | WorkLoc_Office,WorkRemote_Bucket_Presencial / Bajo remoto (<30%),0.145993,0.902639,1.489988
552,WorkChallenge_Meetings | WorkRemote_Bucket_Pre...,"CodeRev_Yes, because I see value in code revie...",0.152910,0.703951,1.477957


### 4.3.1. Primera observación: 6 de 9 reglas sin evidencia directa

De las 9 reglas expertas con contenido relevante para las soft skills, vemos que en **6 de ellas no se encuentra ninguna regla descubierta que las respalde** (#8, #10, #11, #22, #24, #25), y solo hay 3 en las que sí (#12, #23, #26).

A primera vista podemos pensar que el modelo de reglas de asociación no confirma la mayoría del conocimiento experto, pero antes de concluir con ello, vemos que hay un patrón sospechoso en las 6 sin evidencia: **`WorkRemote_Bucket_Remoto Alto (>70%)` aparece en 4 de las 6** (#11, #22, #24, #25), y las otras 2 (#8, #10) comparten `Neuroticismo_Nivel_Alto` combinado con tramos altos de `WorkWeekHrs_Bin`.

Esto nos dice que la ausencia de evidencia podría no deberse a que la relación no exista en los datos, sino a una **limitación del propio proceso de minado**, lo que comprobamos en el siguiente apartado.

### 4.3.2. Diagnóstico: ¿es ausencia de relación o limitación del `min_support`?

Para verificar la hipótesis planteada, vamos a comprobar si cada ítem de las 9 reglas expertas llega a aparecer **en alguna regla del conjunto sin filtrar**. Si uno de ellos no aparece ahí, diremos que **nunca formó parte de ninguna combinación frecuente con otros ítems** bajo `min_support=0.12`, ya que la ausencia de evidencia no dice nada sobre si existe relación real, solo confirma que el ítem es demasiado minoritario en la población para que Apriori lo considere.

In [4]:
reglas_apriori_completas = pd.read_csv(config.REGLAS_APRIORI_PATH)
reglas_apriori_completas['items_totales'] = reglas_apriori_completas.apply(
    lambda row: set(str(row['antecedents']).split(' | ')) | set(str(row['consequents']).split(' | ')),
    axis=1
)

items_a_verificar = set()
for r in reglas_soft:
    items_a_verificar.update(r['items_soft'])

for item in sorted(items_a_verificar):
    aparece = reglas_apriori_completas['items_totales'].apply(lambda s: item in s).any()
    estado = "aparece en alguna regla (aunque sea filtrada después)" if aparece else "❌ NUNCA aparece — no forma ninguna combinación que supere min_support=0.12"
    print(f"{item}: {estado}")

Amabilidad_Nivel_Bajo: aparece en alguna regla (aunque sea filtrada después)
Apertura_Nivel_Bajo: aparece en alguna regla (aunque sea filtrada después)
Neuroticismo_Nivel_Alto: aparece en alguna regla (aunque sea filtrada después)
WorkChallenge_Meetings: aparece en alguna regla (aunque sea filtrada después)
WorkPlan_There's no schedule or spec; I work on what seems most important or urgent: aparece en alguna regla (aunque sea filtrada después)
WorkRemote_Bucket_Remoto Alto (>70%): ❌ NUNCA aparece — no forma ninguna combinación que supere min_support=0.12
WorkWeekHrs_Bin_45-55h: ❌ NUNCA aparece — no forma ninguna combinación que supere min_support=0.12
WorkWeekHrs_Bin_>55h: ❌ NUNCA aparece — no forma ninguna combinación que supere min_support=0.12


De los 8 ítems únicos que componen las 9 reglas expertas, **3 nunca aparecen en ninguna regla** con `min_support=0.12`: `WorkRemote_Bucket_Remoto Alto (>70%)`, `WorkWeekHrs_Bin_45-55h` y `WorkWeekHrs_Bin_>55h`. Los dos tramos de horas no superan el umbral ni siquiera como itemset individual (soporte de 0,075 y 0,035), mientras que `Remoto Alto` lo supera por muy poco (0,1221, coherente con el 12,2% de prevalencia ya visto en el EDA), pero al combinarse con cualquier otro ítem ya no llega al 12% del total, por lo que tampoco forma ninguna regla.

Esto explica exactamente los 6 casos "sin evidencia" de la sección anterior, ya que:
- **#11 y #22** dependen únicamente de `WorkRemote_Bucket_Remoto Alto` y nunca podían tener regla.
- **#24 y #25** también incluyen ese mismo ítem, además de otro.
- **#8 y #10** dependen de los tramos altos de `WorkWeekHrs_Bin`, por debajo del umbral.

Los otros 5 ítems (`Amabilidad_Nivel_Bajo`, `Apertura_Nivel_Bajo`, `Neuroticismo_Nivel_Alto`, `WorkChallenge_Meetings`, `WorkPlan_...no schedule...`) sí superan el umbral y llegan a formar parte de reglas, lo que nos confirma que, para estos, el "sin evidencia" si lo hubiera habido sería un resultado real y no un artefacto del parámetro.

**Como conclusión** podemos decir que la ausencia de evidencia en las 6 reglas (#11, #22, #24, #25, #8 y #10) **no refleja ausencia de relación en los datos**, sino una limitación del `min_support` global elegido en la Libreta 3, apropiado para una primera exploración con 205 ítems, pero demasiado alto para estas categorías minoritarias.

### 4.3.3. Cálculo manual dirigido, sin depender del `min_support` global

Para las 4 reglas afectadas por la limitación de soporte (#8, #10, #24, #25), se calculan soporte, confianza y lift **directamente sobre la matriz transaccional** (`df_matriz`), sin pasar por Apriori. Esto lo hago así para evitar tener que relanzar el algoritmo completo con un umbral más bajo ya que me daría problemas de memoria.

La función `calcular_metricas_manual` reproduce las mismas fórmulas que usa `mlxtend` internamente:
- **Soporte(A)** = proporción de candidatos con el ítem A.
- **Soporte(A∩B)** = proporción de candidatos con ambos ítems a la vez.
- **Confianza(A→B)** = Soporte(A∩B) / Soporte(A) — probabilidad condicional.
- **Lift** = Soporte(A∩B) / (Soporte(A) × Soporte(B)) — fuerza de asociación frente a independencia.

Esta la vamos a aplicar únicamente a los 4 pares de ítems pendientes, no al dataset completo, por lo que el cálculo es inmediato y no reproduce el problema de memoria original.

In [5]:
df_matriz = pd.read_csv(config.MATRIZ_TRANSACCIONAL_PATH, index_col='CandidateId').fillna(0).astype(bool)

In [6]:
def calcular_metricas_manual(item_a, item_b, df=df_matriz):
    n = len(df)
    soporte_a = df[item_a].sum() / n
    soporte_b = df[item_b].sum() / n
    soporte_ab = (df[item_a] & df[item_b]).sum() / n
    confianza = soporte_ab / soporte_a if soporte_a > 0 else None
    lift = soporte_ab / (soporte_a * soporte_b) if soporte_a > 0 and soporte_b > 0 else None
    return {
        'soporte_A': round(soporte_a, 4), 'soporte_B': round(soporte_b, 4),
        'soporte_conjunto': round(soporte_ab, 4), 'confianza_A_to_B': round(confianza, 4) if confianza else None,
        'lift': round(lift, 4) if lift else None, 'n_casos': int((df[item_a] & df[item_b]).sum())}

combinaciones_pendientes = {
    8:  ('Neuroticismo_Nivel_Alto', 'WorkWeekHrs_Bin_>55h'),
    10: ('Neuroticismo_Nivel_Alto', 'WorkWeekHrs_Bin_45-55h'),
    24: ('WorkRemote_Bucket_Remoto Alto (>70%)', 'WorkChallenge_Meetings'),
    25: ('WorkRemote_Bucket_Remoto Alto (>70%)', "WorkPlan_There's no schedule or spec; I work on what seems most important or urgent"),
}

for num, (a, b) in combinaciones_pendientes.items():
    print(f"\nRegla experta #{num}: {a}  ×  {b}")
    print(calcular_metricas_manual(a, b))


Regla experta #8: Neuroticismo_Nivel_Alto  ×  WorkWeekHrs_Bin_>55h
{'soporte_A': np.float64(0.264), 'soporte_B': np.float64(0.0351), 'soporte_conjunto': np.float64(0.02), 'confianza_A_to_B': np.float64(0.0759), 'lift': np.float64(2.1637), 'n_casos': 1778}

Regla experta #10: Neuroticismo_Nivel_Alto  ×  WorkWeekHrs_Bin_45-55h
{'soporte_A': np.float64(0.264), 'soporte_B': np.float64(0.075), 'soporte_conjunto': np.float64(0.0281), 'confianza_A_to_B': np.float64(0.1065), 'lift': np.float64(1.4195), 'n_casos': 2496}

Regla experta #24: WorkRemote_Bucket_Remoto Alto (>70%)  ×  WorkChallenge_Meetings
{'soporte_A': np.float64(0.1221), 'soporte_B': np.float64(0.2812), 'soporte_conjunto': np.float64(0.0426), 'confianza_A_to_B': np.float64(0.3486), 'lift': np.float64(1.2398), 'n_casos': 3779}

Regla experta #25: WorkRemote_Bucket_Remoto Alto (>70%)  ×  WorkPlan_There's no schedule or spec; I work on what seems most important or urgent
{'soporte_A': np.float64(0.1221), 'soporte_B': np.float64(0.2

### 4.4. Interpretación de resultados

| # | Riesgo | Evidencia | Lift | Interpretación |
|---|---|---|---|---|
| 8 | Sobrecarga de trabajo | Cálculo manual (fuera de `min_support`) | 2,16 | **Coincide** — asociación positiva moderada-fuerte entre neuroticismo alto y trabajar >55h/semana, aunque la combinación es rara en términos absolutos (2% del total, 1.778 candidatos). |
| 10 | Burnout (neuroticismo+carga) | Cálculo manual | 1,42 | **Coincide débilmente** — misma dirección que #8 pero con el tramo 45-55h, magnitud más moderada. |
| 11 | Aislamiento social | Prevalencia directa (no requiere combinación) | — | **No aplica test de asociación.** La condición del experto es sobre un único rasgo (remoto >70%), ya cuantificado en el EDA: 12,2% de los candidatos. Confirma que el perfil de riesgo existe en la población, no que derive en aislamiento (no hay variable de resultado en la encuesta). |
| 12 | Conflictos (baja amabilidad) | Apriori (`reglas_limpias`) | 1,41–1,48 | **Evidencia indirecta.** `Amabilidad_Nivel_Bajo` correlaciona con un perfil de trabajo convencional (presencial, jornada completa, horario estándar) — no con conflictos en sí, que la encuesta no puede medir directamente. |
| 22 | Falta de soporte remoto | Prevalencia directa | — | Misma situación que #11 (mismo ítem soft). |
| 23 | Resistencia al cambio | Apriori (`reglas_limpias`) | 1,09–1,34 | **Evidencia indirecta**, mismo patrón que #12: `Apertura_Nivel_Bajo` correlaciona con un perfil de trabajo convencional (jornada completa, 35-45h, presencial) — no con resistencia al cambio en sí. |
| 24 | Fatiga digital | Cálculo manual | 1,24 | **Coincide débilmente** — asociación positiva entre remoto alto y percepción de exceso de reuniones, magnitud modesta. |
| 25 | Burnout (remoto+plazos) | Cálculo manual | 1,45 | **Coincide moderadamente** — remoto alto y ausencia de horario definido se asocian con una fuerza similar a #10. |
| 26 | Burnout (reuniones) | Apriori (`reglas_limpias`) | 1,48–1,53 | **Evidencia indirecta**, mismo patrón que #12: `WorkChallenge_Meetings` correlaciona con perfil convencional de trabajo, no con burnout directamente. |

**Patrón transversal a destacar:** las 4 reglas relacionadas con **burnout/fatiga** (#8, #10, #24, #25) muestran **todas** una asociación positiva (lift >1,2), aunque en general de magnitud modesta — es la familia de riesgos con mejor respaldo cuantitativo de las 9 evaluadas. En cambio, las reglas evaluadas con `reglas_limpias` (#12, #23, #26) solo muestran evidencia indirecta, ya que sus ítems se asocian con el perfil de trabajo convencional y no con el riesgo en sí, posiblemente porque estos riesgos dependen más de dinámicas interpersonales y contextuales que la encuesta no captura bien.

### 4.5. Conclusiones

De las **27 reglas** expertas hemos revisado un total de **9 reglas** utilizando exclusivamente datos de los candidatos de la encuesta de soft skills de 2019. De las 18 restantes, 1 (#6) depende de hard skills y se evalúa en su propio pipeline, y las otras **17** no es posible aplicarlas aquí ya que tratan temas más a nivel de equipo/proyecto (solapamiento horario, tamaño de equipo, políticas de empresa, herramientas...) que no podemos cubrir con los datos de los que disponemos, por lo que se evaluarán en la fase de matching, por comprobación directa contra los campos del JSON de proyecto.

**Resultados más importantes observados:**

1. **Las reglas de burnout/fatiga digital (#8, #10, #24, #25) son las mejor respaldadas por los datos**, con lift consistentemente por encima de 1,2 en las cuatro. Es la familia de riesgos donde el conocimiento experto y el patrón descubierto en los datos convergen con mayor claridad.

2. **Limitación metodológica del `min_support` global**: tres de los ocho ítems evaluables (`WorkRemote_Bucket_Remoto Alto`, `WorkWeekHrs_Bin_45-55h`, `WorkWeekHrs_Bin_>55h`) no forman ninguna combinación que supere el `min_support=0.12` usado en Apriori, por lo que nunca aparecen en ninguna regla minada. Se recalcularon manualmente para no perder esta información, y su ausencia en las reglas minadas se debe al alcance del umbral elegido, no a una limitación oculta.

3. **Limitación estructural de fondo**: ninguna regla descubierta puede validar el *riesgo* en sí (conflictos, resistencia, burnout) porque la encuesta de Stack Overflow no contiene ninguna variable de resultado de ese tipo.

4. Pese a estas limitaciones, la comparación aporta valor real al objetivo del proyecto. Entre otras: identifica qué combinaciones de rasgos de personalidad son estadísticamente consistentes en los datos y cuáles solo cuentan con evidencia indirecta (conflictos, resistencia al cambio y burnout por reuniones), abriendo una discusión sobre los límites de ambos enfoques en lugar de asumir que uno debe primar sobre el otro.